# Notebook to fix PR issues


### Generate the UE data from mobility model first

In [1]:
import sys
from pathlib import Path
sys.path.append(f"{Path().absolute().parent}")

In [2]:
import pandas as pd
import scipy
import numpy as np
from radp_library import *
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from radp.digital_twin.mobility.param_regression import get_predicted_alpha,preprocess_ue_data
from radp.digital_twin.utils.cell_selection import perform_attachment
from radp.digital_twin.rf.bayesian.bayesian_engine import (
    BayesianDigitalTwin,
    NormMethod,
)
from apps.mobility_robustness_optimization.simple_mro import SimpleMRO

/Users/tanzimfarhan/Desktop/Maveric/maveric/.venv/lib/python3.11/site-packages/fastkml/config.py:39: UserWarning: Package `lxml` missing. Pretty print will be disabled
  warnings.warn("Package `lxml` missing. Pretty print will be disabled")  # noqa: B028


In [3]:
params = {
    "ue_tracks_generation": {
            "params": {
                "simulation_duration": 3600,
                "simulation_time_interval_seconds": 0.01,
                "num_ticks": 50,
                "num_batches": 1,
                "ue_class_distribution": {
                    "stationary": {
                        "count": 10,
                        "velocity": 0,
                        "velocity_variance": 1
                    },
                    "pedestrian": {
                        "count": 5,
                        "velocity": 2,
                        "velocity_variance": 1
                    },
                    "cyclist": {
                        "count": 5,
                        "velocity": 5,
                        "velocity_variance": 1
                    },
                    "car": {
                        "count": 12,
                        "velocity": 20,
                        "velocity_variance": 1
                    }
                },
                "lat_lon_boundaries": {
                    "min_lat": -90,
                    "max_lat": 90,
                    "min_lon": -180,
                    "max_lon": 180
                },
                "gauss_markov_params": {
                    "alpha": 0.5,
                    "variance": 0.8,
                    "rng_seed": 42,
                    "lon_x_dims": 100,
                    "lon_y_dims": 100,
                    "// TODO": "Account for supporting the user choosing the anchor_loc and cov_around_anchor.",
                    "// Current implementation": "the UE Tracks generator will not be using these values.",
                    "// anchor_loc": {},
                    "// cov_around_anchor": {}
            }
        }
    }
}

In [4]:
training_data = get_ue_data(params)
training_data.head()

,mock_ue_id,lon,lat,tick
0,0,48.510339,-16.462645,0
1,1,19.286613,63.617111,0
2,2,21.315702,-47.889952,0
3,3,-70.559364,-79.511709,0
4,4,-168.916011,-39.338640,0


In [3]:
topology = pd.read_csv('/Users/tanzimfarhan/Downloads/topology.csv')
topology.loc[topology['cell_id'] == 'cell_2', 'cell_lat'] = 0
topology.loc[topology['cell_id'] == 'cell_3', 'cell_lat'] = 90
topology.loc[topology['cell_id'] == 'cell_1', 'cell_lat'] = -90


topology.loc[topology['cell_id'] == 'cell_2', 'cell_lon'] = 0
topology.loc[topology['cell_id'] == 'cell_3', 'cell_lon'] = 180
topology.loc[topology['cell_id'] == 'cell_1', 'cell_lon'] = -180


topology.loc[topology['cell_id'] == 'cell_2', 'cell_carrier_freq_mhz'] = 2800
topology.loc[topology['cell_id'] == 'cell_3', 'cell_carrier_freq_mhz'] = 2800
topology.loc[topology['cell_id'] == 'cell_1', 'cell_carrier_freq_mhz'] = 2800

In [4]:
def _prepare_all_UEs_from_all_cells_df(
     data, topology
    ) -> pd.DataFrame:
        """
        Connects each user equipment (UE) entry to all cells in the topology for each tick,
        effectively creating a Cartesian product of UEs and cells, which includes data from both sources.
        """

        ue_data = data
        ue_data = ue_data.rename(columns={"lat": "latitude", "lon": "longitude"})
        topology_tmp = topology
        # Remove the 'cell_' prefix and convert cell_id to integer if needed
        if topology_tmp["cell_id"].dtype == object:
            topology_tmp["cell_id"] = (
                topology_tmp["cell_id"].str.replace("cell_", "").astype(int)
            )
        ue_data["key"] = 1
        topology_tmp["key"] = 1
        combined_df = pd.merge(ue_data, topology_tmp, on="key").drop("key", axis=1)
        print("Combined DataFrame:", combined_df)
        return combined_df


In [5]:
def _preprocess_ue_topology_data(data,topology) -> pd.DataFrame:
        full_data = _prepare_all_UEs_from_all_cells_df(data,topology)
        full_data["log_distance"] = full_data.apply(
            lambda row: GISTools.get_log_distance(
                row["latitude"], row["longitude"], row["cell_lat"], row["cell_lon"]
            ),
            axis=1,
        )

        full_data["cell_rxpwr_dbm"] = full_data.apply(
            lambda row: calculate_received_power(
                row["log_distance"], row["cell_carrier_freq_mhz"]
            ),
            axis=1,
        )

        return full_data

In [6]:
def _preprocess_ue_training_data(data,topology) -> pd.DataFrame:
        data = _preprocess_ue_topology_data(data,topology)
        train_per_cell_df = [x for _, x in data.groupby("cell_id")]
        n_cell = len(topology.index)

        metadata_df = pd.DataFrame(
            {
                "cell_id": [cell_id for cell_id in topology.cell_id],
                "idx": [i + 1 for i in range(n_cell)],
            }
        )
        idx_cell_id_mapping = dict(zip(metadata_df.idx, metadata_df.cell_id))
        desired_idxs = [1 + r for r in range(n_cell)]

        n_samples_train = []
        for df in train_per_cell_df:
            n_samples_train.append(df.shape[0])

        train_per_cell_df_processed = []
        for i in range(n_cell):
            train_per_cell_df_processed.append(
                get_percell_data(
                    data_in=train_per_cell_df[i],
                    choose_strongest_samples_percell=False,
                    n_samples=n_samples_train[i],
                )[0][0]
            )

        training_data = {}

        for i, df in enumerate(train_per_cell_df_processed):
            train_cell_id = idx_cell_id_mapping[i + 1]
            training_data[train_cell_id] = df

        for train_cell_id, training_data_idx in training_data.items():
            training_data_idx["cell_id"] = train_cell_id
            training_data_idx["cell_lat"] = topology[
                topology["cell_id"] == train_cell_id
            ]["cell_lat"].values[0]
            training_data_idx["cell_lon"] = topology[
                topology["cell_id"] == train_cell_id
            ]["cell_lon"].values[0]
            training_data_idx["cell_az_deg"] = topology[
                topology["cell_id"] == train_cell_id
            ]["cell_az_deg"].values[0]
            training_data_idx["cell_carrier_freq_mhz"] = topology[
                topology["cell_id"] == train_cell_id
            ]["cell_carrier_freq_mhz"].values[0]
            training_data_idx["relative_bearing"] = [
                GISTools.get_relative_bearing(
                    training_data_idx["cell_az_deg"].values[0],
                    training_data_idx["cell_lat"].values[0],
                    training_data_idx["cell_lon"].values[0],
                    lat,
                    lon,
                )
                for lat, lon in zip(
                    training_data_idx["latitude"], training_data_idx["longitude"]
                )
            ]

        return training_data

In [7]:
bayesian_digital_twins = {}

In [8]:
def _training(bayesian_digital_twins, maxiter: int, train_data: pd.DataFrame,topology: pd.DataFrame) -> List[float]:
        """
        Trains the Bayesian Digital Twins for each cell in the topology using the UE locations and features
        like log distance, relative bearing, and cell received power (Rx power).
        """
        training_data = _preprocess_ue_training_data(train_data,topology)
        loss_vs_iters = []
        for train_cell_id, training_data_idx in training_data.items():
            bayesian_digital_twins[train_cell_id] = BayesianDigitalTwin(
                data_in=[training_data_idx],
                x_columns=["log_distance", "relative_bearing"],
                y_columns=["cell_rxpwr_dbm"],
                norm_method=NormMethod.MINMAX,
            )
            bayesian_digital_twins[train_cell_id] = bayesian_digital_twins[
                train_cell_id
            ]
            loss_vs_iters.append(
                bayesian_digital_twins[train_cell_id].train_distributed_gpmodel(
                    maxiter=maxiter,
                )
            )
        return bayesian_digital_twins, loss_vs_iters

In [11]:
bayesian_digital_twins, loss_vs_iters = _training(
    bayesian_digital_twins,
    maxiter=100,
    train_data=training_data,
    topology=topology,
)


Combined DataFrame:       mock_ue_id   longitude   latitude  tick  cell_lat  cell_lon  cell_id  \
0              0   48.510339 -16.462645     0     -90.0    -180.0        1   
1              0   48.510339 -16.462645     0       0.0       0.0        2   
2              0   48.510339 -16.462645     0      90.0     180.0        3   
3              1   19.286613  63.617111     0     -90.0    -180.0        1   
4              1   19.286613  63.617111     0       0.0       0.0        2   
...          ...         ...        ...   ...       ...       ...      ...   
4795          30   89.100351 -11.651079    49       0.0       0.0        2   
4796          30   89.100351 -11.651079    49      90.0     180.0        3   
4797          31  146.478589  74.187621    49     -90.0    -180.0        1   
4798          31  146.478589  74.187621    49       0.0       0.0        2   
4799          31  146.478589  74.187621    49      90.0     180.0        3   

      cell_az_deg  cell_carrier_freq_mhz  


[2025-05-06 22:20:05,799] INFO:  Iter 1/100 - Loss: 0.771 (delta=inf)
[2025-05-06 22:20:05,824] INFO:  Iter 2/100 - Loss: 0.752 (delta=-0.018841)
[2025-05-06 22:20:05,849] INFO:  Iter 3/100 - Loss: 0.731 (delta=-0.020256)
[2025-05-06 22:20:05,880] INFO:  Iter 4/100 - Loss: 0.713 (delta=-0.018046)
[2025-05-06 22:20:05,919] INFO:  Iter 5/100 - Loss: 0.695 (delta=-0.018767)
[2025-05-06 22:20:05,953] INFO:  Iter 6/100 - Loss: 0.676 (delta=-0.018556)
[2025-05-06 22:20:05,989] INFO:  Iter 7/100 - Loss: 0.658 (delta=-0.017709)
[2025-05-06 22:20:06,023] INFO:  Iter 8/100 - Loss: 0.635 (delta=-0.023394)
[2025-05-06 22:20:06,058] INFO:  Iter 9/100 - Loss: 0.615 (delta=-0.019590)
[2025-05-06 22:20:06,088] INFO:  Iter 10/100 - Loss: 0.600 (delta=-0.015070)
[2025-05-06 22:20:06,120] INFO:  Iter 11/100 - Loss: 0.573 (delta=-0.027165)
[2025-05-06 22:20:06,164] INFO:  Iter 12/100 - Loss: 0.558 (delta=-0.014900)
[2025-05-06 22:20:06,194] INFO:  Iter 13/100 - Loss: 0.535 (delta=-0.022887)
[2025-05-06 22

In [12]:
bayesian_digital_twins

{1: <radp.digital_twin.rf.bayesian.bayesian_engine.BayesianDigitalTwin at 0x17da01890>,
 2: <radp.digital_twin.rf.bayesian.bayesian_engine.BayesianDigitalTwin at 0x3038425d0>,
 3: <radp.digital_twin.rf.bayesian.bayesian_engine.BayesianDigitalTwin at 0x3044d3750>}

In [22]:
params2 = {
    "ue_tracks_generation": {
            "params": {
                "simulation_duration": 3600,
                "simulation_time_interval_seconds": 0.01,
                "num_ticks": 50,
                "num_batches": 1,
                "ue_class_distribution": {
                    "stationary": {
                        "count": 10,
                        "velocity": 0,
                        "velocity_variance": 1
                    },
                    "pedestrian": {
                        "count": 5,
                        "velocity": 2,
                        "velocity_variance": 1
                    },
                    "cyclist": {
                        "count": 7,
                        "velocity": 5,
                        "velocity_variance": 1
                    },
                    "car": {
                        "count": 10,
                        "velocity": 20,
                        "velocity_variance": 1
                    }
                },
                "lat_lon_boundaries": {
                    "min_lat": -90,
                    "max_lat": 90,
                    "min_lon": -180,
                    "max_lon": 180
                },
                "gauss_markov_params": {
                    "alpha": 0.8,
                    "variance": 0.5,
                    "rng_seed": 42,
                    "lon_x_dims": 100,
                    "lon_y_dims": 100,
                    "// TODO": "Account for supporting the user choosing the anchor_loc and cov_around_anchor.",
                    "// Current implementation": "the UE Tracks generator will not be using these values.",
                    "// anchor_loc": {},
                    "// cov_around_anchor": {}
            }
        }
    }
}

In [23]:
def _preprocess_prediction_data(pred_data,topology) -> pd.DataFrame:
        data = _prepare_all_UEs_from_all_cells_df(pred_data,topology)

        data["log_distance"] = data.apply(
            lambda row: GISTools.get_log_distance(
                row["latitude"], row["longitude"], row["cell_lat"], row["cell_lon"]
            ),
            axis=1,
        )
        data["cell_rxpwr_dbm"] = data.apply(
            lambda row: calculate_received_power(
                row["log_distance"], row["cell_carrier_freq_mhz"]
            ),
            axis=1,
        )

        data["relative_bearing"] = data.apply(
            lambda row: GISTools.get_relative_bearing(
                row["cell_az_deg"],
                row["cell_lat"],
                row["cell_lon"],
                row["latitude"],
                row["longitude"],
            ),
            axis=1,
        )
        return data

In [24]:
# Prediction
def _predictions(pred_data,topology,bayesian_digital_twins) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """
        Predicts the received power for each User Equipment (UE) at different locations and ticks using Bayesian Digital Twins.
        It then determines the best cell for each UE to attach based on the predicted power values.
        """
        prediction_data = _preprocess_prediction_data(pred_data,topology)
        full_prediction_df = pd.DataFrame()

        # Loop over each 'tick'
        for tick, tick_df in prediction_data.groupby("tick"):
            # Loop over each 'cell_id' within the current 'tick'
            for cell_id, cell_df in tick_df.groupby("cell_id"):
                # Check if the Bayesian model for this cell_id exists
                if cell_id in bayesian_digital_twins:
                    # Perform the Bayesian prediction
                    pred_means_percell, _ = bayesian_digital_twins[
                        cell_id
                    ].predict_distributed_gpmodel(prediction_dfs=[cell_df])

                    # Assuming 'pred_means_percell' returns a list of predictions corresponding to the DataFrame index
                    cell_df["pred_means"] = pred_means_percell[0]

                    # Include additional necessary columns for the final DataFrame
                    cell_df["tick"] = tick
                    cell_df["cell_id"] = cell_id

                    # Append the predictions to the full DataFrame
                    full_prediction_df = pd.concat(
                        [full_prediction_df, cell_df], ignore_index=True
                    )
                else:
                    # Handle missing models, e.g., log a warning or initialize a default model
                    print(
                        f"No model available for cell_id {cell_id}, skipping prediction."
                    )

        full_prediction_df = full_prediction_df.rename(
            columns={"latitude": "loc_y", "longitude": "loc_x"}
        )
        predicted = perform_attachment(full_prediction_df, topology)

        return predicted, full_prediction_df

In [25]:
prediction_data = get_ue_data(params2)

In [26]:
predictions, full_prediction_df = _predictions(
    pred_data=prediction_data,
    topology=topology,
    bayesian_digital_twins = bayesian_digital_twins
)

Combined DataFrame:       mock_ue_id   longitude   latitude  tick  cell_lat  cell_lon  cell_id  \
0              0   48.510339 -16.462645     0     -90.0    -180.0        1   
1              0   48.510339 -16.462645     0       0.0       0.0        2   
2              0   48.510339 -16.462645     0      90.0     180.0        3   
3              1   19.286613  63.617111     0     -90.0    -180.0        1   
4              1   19.286613  63.617111     0       0.0       0.0        2   
...          ...         ...        ...   ...       ...       ...      ...   
4795          30  -64.060471  70.447505    49       0.0       0.0        2   
4796          30  -64.060471  70.447505    49      90.0     180.0        3   
4797          31 -152.742476  51.756565    49     -90.0    -180.0        1   
4798          31 -152.742476  51.756565    49       0.0       0.0        2   
4799          31 -152.742476  51.756565    49      90.0     180.0        3   

      cell_az_deg  cell_carrier_freq_mhz  


/Users/tanzimfarhan/Desktop/Maveric/maveric/.venv/lib/python3.11/site-packages/gpytorch/distributions/multivariate_normal.py:319: NumericalWarning: Negative variance values detected. This is likely due to numerical instabilities. Rounding negative variances up to 1e-06.
  warnings.warn(
/Users/tanzimfarhan/Desktop/Maveric/maveric/.venv/lib/python3.11/site-packages/gpytorch/distributions/multivariate_normal.py:319: NumericalWarning: Negative variance values detected. This is likely due to numerical instabilities. Rounding negative variances up to 1e-06.
  warnings.warn(
/Users/tanzimfarhan/Desktop/Maveric/maveric/.venv/lib/python3.11/site-packages/gpytorch/distributions/multivariate_normal.py:319: NumericalWarning: Negative variance values detected. This is likely due to numerical instabilities. Rounding negative variances up to 1e-06.
  warnings.warn(
/Users/tanzimfarhan/Desktop/Maveric/maveric/.venv/lib/python3.11/site-packages/gpytorch/distributions/multivariate_normal.py:319: Numeri

### Disect all the codes from MRO and put it to radp_library.py


In [9]:
bayesian_digital_twins = {}

In [10]:
update_data = pd.read_csv('/Users/tanzimfarhan/Downloads/combined_data_dump.csv')
update_data

,latitude,longitude,cell_rxpower_dbm
0,59.806764,-22.625309,100.788266
1,59.806764,-22.625309,100.788266
2,59.806764,-22.625309,100.788266
3,54.857584,119.764151,99.608310
4,54.857584,119.764151,99.608310
...,...,...,...
5995,-26.656397,-16.480045,99.734971
5996,-26.656397,-16.480045,99.734971
5997,63.970820,34.222834,100.191265
5998,63.970820,34.222834,100.191265


In [11]:
def _preprocess_ue_update_data(update_data,topology) -> pd.DataFrame:
        data = _prepare_all_UEs_from_all_cells_df(update_data,topology)
        data["log_distance"] = data.apply(
            lambda row: GISTools.get_log_distance(
                row["latitude"], row["longitude"], row["cell_lat"], row["cell_lon"]
            ),
            axis=1,
        )

        update_per_cell_df = [x for _, x in data.groupby("cell_id")]
        n_cell = len(topology.index)

        metadata_df = pd.DataFrame(
            {
                "cell_id": [cell_id for cell_id in topology.cell_id],
                "idx": [i + 1 for i in range(n_cell)],
            }
        )
        idx_cell_id_mapping = dict(zip(metadata_df.idx, metadata_df.cell_id))

        n_samples_update = []
        for df in update_per_cell_df:
            n_samples_update.append(df.shape[0])

        update_per_cell_df_processed = []
        for i in range(n_cell):
            update_per_cell_df_processed.append(
                get_percell_data(
                    data_in=update_per_cell_df[i],
                    choose_strongest_samples_percell=False,
                    n_samples=n_samples_update[i],
                )[0][0]
            )

        update_data = {}

        for i, df in enumerate(update_per_cell_df_processed):
            update_cell_id = idx_cell_id_mapping[i + 1]
            update_data[update_cell_id] = df

        for update_cell_id, update_data_idx in update_data.items():
            update_data_idx["cell_id"] = update_cell_id
            update_data_idx["cell_lat"] = topology[
                topology["cell_id"] == update_cell_id
            ]["cell_lat"].values[0]
            update_data_idx["cell_lon"] = topology[
                topology["cell_id"] == update_cell_id
            ]["cell_lon"].values[0]
            update_data_idx["cell_az_deg"] = topology[
                topology["cell_id"] == update_cell_id
            ]["cell_az_deg"].values[0]
            update_data_idx["cell_carrier_freq_mhz"] = topology[
                topology["cell_id"] == update_cell_id
            ]["cell_carrier_freq_mhz"].values[0]
            update_data_idx["relative_bearing"] = [
                GISTools.get_relative_bearing(
                    update_data_idx["cell_az_deg"].values[0],
                    update_data_idx["cell_lat"].values[0],
                    update_data_idx["cell_lon"].values[0],
                    lat,
                    lon,
                )
                for lat, lon in zip(
                    update_data_idx["latitude"], update_data_idx["longitude"]
                )
            ]
        return update_data

In [16]:
def train_or_update_rf_twin(new_data: pd.DataFrame,topology: pd.DataFrame, bayesian_digital_twins):
        try:
            if not isinstance(new_data, pd.DataFrame):
                raise TypeError("The input 'new_data' must be a pandas DataFrame.")
            
            new_data = new_data.rename(
                columns={"cell_rxpower_dbm": "cell_rxpwr_dbm"}
            )
            expected_columns = {"longitude", "latitude", "cell_rxpwr_dbm"}
            if not expected_columns.issubset(new_data.columns):
                raise ValueError(
                    f"The input DataFrame must contain the following columns: {expected_columns}"
                )

            if bayesian_digital_twins:
                update_data = new_data
                updated_data = _preprocess_ue_update_data(update_data, topology)
                updated_data_list = list(updated_data.values())
                print("Updated_List ",updated_data_list)

                for data_idx, update_data_df in enumerate(updated_data_list):
                    update_cell_id = data_idx + 1
                    if update_cell_id in bayesian_digital_twins:
                        bayesian_digital_twins[
                            update_cell_id
                        ].update_trained_gpmodel([update_data_df])
            else:
                print(
                    "No Bayesian Digital Twins available for update. Training from scratch."
                )
                new_data = new_data.drop(
                    columns=["cell_rxpower_dbm"], errors="ignore"
                )
                _training(bayesian_digital_twins,maxiter=100, train_data=new_data,topology = topology)
        except TypeError as te:
            print(f"TypeError: {te}")
        except ValueError as ve:
            print(f"ValueError: {ve}")
        except KeyError as ke:
            print(f"KeyError: {ke}")
        except Exception as e:
            print(f"An unexpected error occurred: {e}")

In [13]:
updates = train_or_update_rf_twin(update_data,topology,bayesian_digital_twins)

No Bayesian Digital Twins available for update. Training from scratch.
Combined DataFrame:         latitude  longitude  cell_rxpwr_dbm  cell_lat  cell_lon  cell_id  \
0      59.806764 -22.625309      100.788266     -90.0    -180.0        1   
1      59.806764 -22.625309      100.788266       0.0       0.0        2   
2      59.806764 -22.625309      100.788266      90.0     180.0        3   
3      59.806764 -22.625309      100.788266     -90.0    -180.0        1   
4      59.806764 -22.625309      100.788266       0.0       0.0        2   
...          ...        ...             ...       ...       ...      ...   
17995  63.970820  34.222834      100.191265       0.0       0.0        2   
17996  63.970820  34.222834      100.191265      90.0     180.0        3   
17997  63.970820  34.222834      100.191265     -90.0    -180.0        1   
17998  63.970820  34.222834      100.191265       0.0       0.0        2   
17999  63.970820  34.222834      100.191265      90.0     180.0        3 

[2025-05-06 22:49:35,365] INFO:  Iter 1/100 - Loss: 0.758 (delta=inf)
[2025-05-06 22:49:35,846] INFO:  Iter 2/100 - Loss: 0.739 (delta=-0.018518)
[2025-05-06 22:49:36,264] INFO:  Iter 3/100 - Loss: 0.721 (delta=-0.018669)
[2025-05-06 22:49:36,694] INFO:  Iter 4/100 - Loss: 0.702 (delta=-0.018836)
[2025-05-06 22:49:37,108] INFO:  Iter 5/100 - Loss: 0.683 (delta=-0.019022)
[2025-05-06 22:49:37,519] INFO:  Iter 6/100 - Loss: 0.664 (delta=-0.019220)
[2025-05-06 22:49:38,073] INFO:  Iter 7/100 - Loss: 0.644 (delta=-0.019418)
[2025-05-06 22:49:38,499] INFO:  Iter 8/100 - Loss: 0.625 (delta=-0.019617)
[2025-05-06 22:49:38,904] INFO:  Iter 9/100 - Loss: 0.605 (delta=-0.019824)
[2025-05-06 22:49:39,318] INFO:  Iter 10/100 - Loss: 0.585 (delta=-0.020026)
[2025-05-06 22:49:39,877] INFO:  Iter 11/100 - Loss: 0.565 (delta=-0.020224)
[2025-05-06 22:49:40,282] INFO:  Iter 12/100 - Loss: 0.544 (delta=-0.020427)
[2025-05-06 22:49:40,687] INFO:  Iter 13/100 - Loss: 0.523 (delta=-0.020639)
[2025-05-06 22

In [18]:
bayesian_digital_twins

{1: <radp.digital_twin.rf.bayesian.bayesian_engine.BayesianDigitalTwin at 0x15ab24c10>,
 2: <radp.digital_twin.rf.bayesian.bayesian_engine.BayesianDigitalTwin at 0x169bf2f50>,
 3: <radp.digital_twin.rf.bayesian.bayesian_engine.BayesianDigitalTwin at 0x15f3920d0>}

In [27]:
# Training from scratch done now will update the exisitng BDT
update_not_from_scratch = train_or_update_rf_twin(update_data,topology,bayesian_digital_twins)

Combined DataFrame:         latitude  longitude  cell_rxpwr_dbm  cell_lat  cell_lon  cell_id  \
0      59.806764 -22.625309      100.788266     -90.0    -180.0        1   
1      59.806764 -22.625309      100.788266       0.0       0.0        2   
2      59.806764 -22.625309      100.788266      90.0     180.0        3   
3      59.806764 -22.625309      100.788266     -90.0    -180.0        1   
4      59.806764 -22.625309      100.788266       0.0       0.0        2   
...          ...        ...             ...       ...       ...      ...   
17995  63.970820  34.222834      100.191265       0.0       0.0        2   
17996  63.970820  34.222834      100.191265      90.0     180.0        3   
17997  63.970820  34.222834      100.191265     -90.0    -180.0        1   
17998  63.970820  34.222834      100.191265       0.0       0.0        2   
17999  63.970820  34.222834      100.191265      90.0     180.0        3   

       cell_az_deg  cell_carrier_freq_mhz  
0                0     

/Users/tanzimfarhan/Desktop/Maveric/maveric/.venv/lib/python3.11/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-06 to the diagonal
  warnings.warn(
/Users/tanzimfarhan/Desktop/Maveric/maveric/.venv/lib/python3.11/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-05 to the diagonal
  warnings.warn(
/Users/tanzimfarhan/Desktop/Maveric/maveric/.venv/lib/python3.11/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-04 to the diagonal
  warnings.warn(


An unexpected error occurred: Matrix not positive definite after repeatedly adding jitter up to 1.0e-04.
